# 🎨 PhotoEnhance — AI Photo Upscaler
**Безкоштовне покращення якості фото за допомогою Real-ESRGAN**

---

| Параметр | Значення |
|---|---|
| Модель | Real-ESRGAN |
| GPU | Google Colab T4 (16GB) |
| Масштаб | x2, x4, x8 |
| Формати | JPG, PNG, WEBP |
| Вартість | $0 |

> ⚠️ **Переконайся, що увімкнено GPU:** Runtime → Change runtime type → T4 GPU

## Як користуватись: 3 кроки
| Крок | Що робить |
|---|---|
| ▶ **Крок 1** | Встановлення, KeepAlive, перевірка GPU |
| ▶ **Крок 2** | Google Drive + завантаження фото |
| ▶ **Крок 3** | Обробка та скачування результату |


In [ ]:
# @title ⚙️ Крок 1: Встановлення, KeepAlive та перевірка GPU { display-mode: "form" }
# @markdown Запускай **один раз** на початку сесії.

import sys, os

# ════════════════════════════════════════════════
# 🔎 ПЕРЕВІРКИ ПЕРЕД КРОКОМ 1
# ════════════════════════════════════════════════
_ok = True

# 1) Python версія
_ver = sys.version_info
if _ver < (3, 8):
    print(f'❌ Python {_ver.major}.{_ver.minor} — потрібен 3.8+. Зміни runtime у Colab.')
    _ok = False
else:
    print(f'✅ Python {_ver.major}.{_ver.minor}.{_ver.micro}')

# 2) Середовище — Colab?
try:
    import google.colab
    print('✅ Середовище: Google Colab')
except ImportError:
    print('⚠️  Не Colab — деякі функції (Drive, files.download) можуть не працювати.')

# 3) Доступна RAM
try:
    with open('/proc/meminfo') as _m:
        _ram_kb = int([l for l in _m if 'MemAvailable' in l][0].split()[1])
    _ram_gb = _ram_kb / 1024**2
    if _ram_gb < 2:
        print(f'⚠️  Мало RAM: {_ram_gb:.1f} GB вільно — можливі помилки при обробці великих фото.')
    else:
        print(f'✅ RAM доступно: {_ram_gb:.1f} GB')
except Exception:
    print('ℹ️  RAM: не вдалося перевірити')

# 4) Місце на диску /content
try:
    import shutil as _sh
    _total, _used, _free = _sh.disk_usage('/content')
    _free_gb = _free / 1024**3
    if _free_gb < 3:
        print(f'⚠️  Мало місця на диску: {_free_gb:.1f} GB — може не вистачити для моделі (~300 MB) та результатів.')
    else:
        print(f'✅ Місце на диску: {_free_gb:.1f} GB вільно')
except Exception:
    print('ℹ️  Диск: не вдалося перевірити')

if not _ok:
    raise SystemExit('❌ Усунь помилки вище перед запуском Кроку 1.')

print('\n📦 Всі перевірки пройдено — починаємо встановлення...\n')

# ════════════════════════════════════════════════
# 1A. ВСТАНОВЛЕННЯ ЗАЛЕЖНОСТЕЙ
# ════════════════════════════════════════════════
print('📦 Встановлення залежностей...')

!pip install -q basicsr facexlib gfpgan
!pip install -q realesrgan
!pip install -q ipywidgets Pillow opencv-python-headless

# Фікс сумісності basicsr з torchvision >= 0.16
import sys
from types import ModuleType
import torchvision.transforms.functional as _tvf

_compat = ModuleType('torchvision.transforms.functional_tensor')
_compat.rgb_to_grayscale = _tvf.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = _compat

import os
if not os.path.exists('/content/Real-ESRGAN'):
    !git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN
    %cd /content/Real-ESRGAN
    !pip install -q -r requirements.txt
    !python setup.py develop -q
else:
    %cd /content/Real-ESRGAN

print('✅ Залежності встановлено!\n')

# ════════════════════════════════════════════════
# 1B. KEEPALIVE — захист від розриву з'єднання
# ════════════════════════════════════════════════
import threading
from IPython.display import display, Javascript

display(Javascript("""
(function startKeepAlive() {
    if (window._peKeepAlive) clearInterval(window._peKeepAlive);
    window._peKeepAlive = setInterval(function() {
        fetch('/api/kernels').catch(function(){});
    }, 45000);
    console.log('[PhotoEnhance] KeepAlive started — ping every 45s');
})();
"""))

_ka_stop = threading.Event()
def _heartbeat():
    c = 0
    while not _ka_stop.wait(timeout=50):
        c += 1
threading.Thread(target=_heartbeat, daemon=True, name='PE-KeepAlive').start()
print('🔌 KeepAlive активовано (45 сек браузер / 50 сек kernel)\n')

# ════════════════════════════════════════════════
# 1C. ПЕРЕВІРКА GPU
# ════════════════════════════════════════════════
import subprocess
import torch

print('=' * 50)
print('🖥️  ІНФОРМАЦІЯ ПРО GPU')
print('=' * 50)

if torch.cuda.is_available():
    gpu_name   = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU знайдено: {gpu_name}')
    print(f'💾 VRAM: {gpu_memory:.1f} GB')
    print(f'🔧 CUDA версія: {torch.version.cuda}')
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free,temperature.gpu',
         '--format=csv,noheader,nounits'], capture_output=True, text=True)
    if result.returncode == 0:
        parts = result.stdout.strip().split(', ')
        if len(parts) >= 4:
            print(f'🌡️  Температура: {parts[3]}°C')
            print(f'📊 Вільна пам\'ять: {int(parts[2])//1024:.1f} GB / {int(parts[1])//1024:.1f} GB')
    print('\n✅ Все готово! Переходь до Кроку 2 ▶')
else:
    print('❌ GPU не знайдено!')
    print('👉 Runtime → Change runtime type → T4 GPU → Save → перезапусти Крок 1')

print('=' * 50)


In [ ]:
# @title 🖼️ Крок 2: Google Drive + Завантаження фото { display-mode: "form" }
# @markdown **Drive** — для автозбереження результатів. Можна пропустити — файл скачається напряму у Кроці 3.
# @markdown
# @markdown 💡 Потрібен особистий **@gmail.com**. 📱 На телефоні: якщо авторизація не йде — просто продовжуй.

import sys, os

# ════════════════════════════════════════════════
# 🔎 ПЕРЕВІРКИ ПЕРЕД КРОКОМ 2
# ════════════════════════════════════════════════
_ok = True

# 1) Крок 1 виконано? (torch має бути імпортовано)
if 'torch' not in sys.modules:
    print('❌ Крок 1 не виконано! Запусти Крок 1 і дочекайся "✅ Все готово".')
    _ok = False
else:
    print('✅ Крок 1 виконано')

# 2) GPU доступний?
if 'torch' in sys.modules:
    import torch as _t
    if not _t.cuda.is_available():
        print('⚠️  GPU не знайдено — обробка буде дуже повільною (10+ хв).')
        print('   👉 Runtime → Change runtime type → T4 GPU → Save → перезапусти з Кроку 1.')
    else:
        _vram = _t.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'✅ GPU: {_t.cuda.get_device_name(0)} ({_vram:.1f} GB VRAM)')
        if _vram < 4:
            print('⚠️  VRAM < 4 GB — при x8 масштабі може виникнути OOM. Використовуй x2 або x4.')

# 3) Підтримувані бібліотеки встановлено?
for _lib in ['realesrgan', 'basicsr', 'cv2', 'PIL']:
    try:
        __import__(_lib)
        print(f'✅ {_lib} встановлено')
    except ImportError:
        print(f'❌ {_lib} не знайдено — перезапусти Крок 1.')
        _ok = False

# 4) Місце на диску
try:
    import shutil as _sh
    _, _, _free = _sh.disk_usage('/content')
    if _free / 1024**3 < 1:
        print('⚠️  Менше 1 GB на диску — може не вистачити для збереження результату.')
    else:
        print(f'✅ Диск: {_free/1024**3:.1f} GB вільно')
except Exception:
    pass

if not _ok:
    raise SystemExit('❌ Усунь помилки вище перед запуском Кроку 2.')

print('\n✅ Всі перевірки пройдено!\n')

from google.colab import drive
import os
import io
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from PIL import Image

# ════════════════════════════════════════════════
# ЛІМІТИ (змінюй тут)
# ════════════════════════════════════════════════
MAX_FILE_MB = 15        # Максимальний розмір файлу в MB
PROCESS_TIMEOUT = 30   # Максимальний час обробки в секундах

# Максимальна кількість пікселів по масштабу (для вкладення в ~30 сек)
MAX_PIXELS = {2: 6_000_000, 4: 3_000_000, 8: 800_000}

TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

# ════════════════════════════════════════════════
# 2A. GOOGLE DRIVE
# ════════════════════════════════════════════════
if os.path.isdir('/content/drive/MyDrive'):
    print('✅ Google Drive вже підключено.')
    OUTPUT_DIR = '/content/drive/MyDrive/PhotoEnhance_Results'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    DRIVE_AVAILABLE = True
    print(f'📂 Збереження: {OUTPUT_DIR}\n')
else:
    try:
        print('🔗 Підключення Google Drive...')
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/PhotoEnhance_Results'
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        DRIVE_AVAILABLE = True
        print(f'✅ Drive підключено: {OUTPUT_DIR}\n')
    except Exception as e:
        print(f'⚠️  Drive недоступний: {e}')
        print('   Файл скачається напряму у Кроці 3.\n')
        OUTPUT_DIR = TEMP_DIR
        DRIVE_AVAILABLE = False

# ════════════════════════════════════════════════
# 2B. АДАПТИВНІ СТИЛІ (мобільні)
# ════════════════════════════════════════════════
display(HTML("""
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0">
<style>
  .pe-title { font-family: sans-serif; color: #1a1a2e; margin-bottom: 10px; }
  .pe-info  { font-family: monospace; background: #f0f4ff; padding: 10px;
               border-radius: 6px; border-left: 4px solid #4a90e2;
               word-break: break-word; font-size: 13px; }
  .pe-error { font-family: sans-serif; background: #fff0f0; padding: 10px;
               border-radius: 6px; border-left: 4px solid #e74c3c; font-size: 13px; }
  .pe-warn  { font-family: sans-serif; background: #fffbe6; padding: 10px;
               border-radius: 6px; border-left: 4px solid #f39c12; font-size: 13px; }
  .widget-button { min-height: 48px !important; font-size: 15px !important; }
  @media (max-width: 600px) {
    .widget-select select, .widget-dropdown select { width: 100% !important; font-size: 14px !important; }
    .widget-upload, .widget-button { width: 100% !important; min-height: 52px !important; font-size: 16px !important; }
    .widget-toggle-buttons .widget-toggle-button { font-size: 11px !important; padding: 4px 6px !important; }
    .widget-hbox { flex-wrap: wrap !important; gap: 6px !important; }
    .widget-vbox { width: 100% !important; max-width: 100% !important; }
  }
</style>
"""))

# ════════════════════════════════════════════════
# 2C. ІНТЕРФЕЙС ЗАВАНТАЖЕННЯ
# ════════════════════════════════════════════════
title_html = widgets.HTML(f"""
<div class="pe-title">
  <h2>🎨 PhotoEnhance MVP</h2>
  <p style="color:#666;">Безкоштовне AI покращення якості фото — powered by Real-ESRGAN</p>
  <p style="color:#888;font-size:12px;">
    📏 Ліміт файлу: <b>{MAX_FILE_MB} MB</b> &nbsp;|&nbsp;
    ⏱️ Таймаут обробки: <b>{PROCESS_TIMEOUT} сек</b> &nbsp;|&nbsp;
    📐 Формати: JPG, PNG, WEBP
  </p>
</div>
""")

upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp', multiple=False,
    description='📸 Вибрати фото',
    layout=widgets.Layout(width='auto', min_width='200px', min_height='48px')
)

scale_options = widgets.ToggleButtons(
    options=[('× 2  (швидше)', 2), ('× 4  (оптимально)', 4), ('× 8  (max якість)', 8)],
    value=4, description='Масштаб:',
    style={'button_width': 'auto'}
)

model_options = widgets.Dropdown(
    options=[
        ('🖼️  Загальні фото (реальні сцени)', 'RealESRGAN_x4plus'),
        ('🎌 Аніме / Ілюстрації', 'RealESRGAN_x4plus_anime_6B'),
        ('📷 Старі / деградовані фото', 'ESRGAN_SRx4_DF2KOST'),
        ('⚡ Швидкий режим (x2)', 'RealESRGAN_x2plus'),
    ],
    value='RealESRGAN_x4plus', description='Модель:',
    layout=widgets.Layout(width='100%', max_width='460px')
)

preview_output = widgets.Output()
info_label = widgets.HTML('<div class="pe-info">📁 Фото не вибрано</div>')

def on_upload_change(change):
    if upload_btn.value:
        for fname, fdata in upload_btn.value.items():
            size_kb = len(fdata['content']) / 1024
            size_mb = size_kb / 1024

            # ── Перевірка розміру файлу ──
            if size_mb > MAX_FILE_MB:
                info_label.value = (
                    f'<div class="pe-error">'
                    f'❌ <b>Файл занадто великий: {size_mb:.1f} MB</b><br>'
                    f'Максимальний розмір: <b>{MAX_FILE_MB} MB</b>.<br>'
                    f'Стисни фото або обери інше.'
                    f'</div>'
                )
                with preview_output:
                    clear_output()
                return

            # ── Перевірка, що це зображення ──
            try:
                img = Image.open(io.BytesIO(fdata['content']))
            except Exception:
                info_label.value = '<div class="pe-error">❌ Файл не є зображенням — обери JPG/PNG/WEBP.</div>'
                with preview_output:
                    clear_output()
                return

            w, h = img.size
            total_px = w * h

            # ── Попередження про великі розміри ──
            scale_val = scale_options.value
            max_px = MAX_PIXELS.get(scale_val, 3_000_000)
            if total_px > max_px:
                resize_note = (
                    f'<br>⚙️ Фото буде авто-зменшено до ~{max_px//1_000_000:.0f} Mpx '
                    f'перед обробкою (ліміт для x{scale_val} = 30 сек).'
                )
            else:
                resize_note = ''

            info_label.value = (
                f'<div class="pe-info">'
                f'✅ <b>{fname}</b><br>'
                f'📐 {w}×{h} px &nbsp;|&nbsp; {size_kb:.0f} KB &nbsp;|&nbsp; {img.mode}'
                f'{resize_note}'
                f'</div>'
            )

            with preview_output:
                clear_output(wait=True)
                img_thumb = img.copy()
                img_thumb.thumbnail((280, 280))
                import matplotlib.pyplot as plt
                fig, ax = plt.subplots(figsize=(3.5, 3.5))
                ax.imshow(img_thumb)
                ax.set_title('Оригінал (прев\'ю)', fontsize=10)
                ax.axis('off')
                plt.tight_layout()
                plt.show()

upload_btn.observe(on_upload_change, names='value')

ui = widgets.VBox([
    title_html,
    upload_btn, info_label, preview_output,
    widgets.HTML('<hr>'),
    model_options, scale_options,
    widgets.HTML('<hr style="margin-bottom:6px">'),
    widgets.HTML('<b>✅ Готово? Запускай Крок 3 ▶</b>'),
], layout=widgets.Layout(width='100%', max_width='600px'))

display(ui)


In [ ]:
# @title ⚡ Крок 3: Обробка фото + Результат + Скачування { display-mode: "form" }
# @markdown Запускай після вибору фото у Кроці 2.

import os, io, time, urllib.request, sys, shutil, threading, math, base64, re
from types import ModuleType

# Фікс torchvision сумісності
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    import torchvision.transforms.functional as _tvf
    _compat = ModuleType('torchvision.transforms.functional_tensor')
    _compat.rgb_to_grayscale = _tvf.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _compat

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, HTML, clear_output

# Фікс PIL DecompressionBombError для великих зображень
Image.MAX_IMAGE_PIXELS = None

# ════════════════════════════════════════════════
# ЛІМІТИ (повинні збігатись з Кроком 2)
# ════════════════════════════════════════════════
_max_file_mb  = MAX_FILE_MB      if 'MAX_FILE_MB'      in dir() else 15
_max_pixels   = MAX_PIXELS       if 'MAX_PIXELS'       in dir() else {2: 6_000_000, 4: 3_000_000, 8: 800_000}
_proc_timeout = PROCESS_TIMEOUT  if 'PROCESS_TIMEOUT'  in dir() else 30

# ════════════════════════════════════════════════
# 3A. КОНФІГУРАЦІЯ МОДЕЛЕЙ
# ════════════════════════════════════════════════
MODEL_URLS = {
    'RealESRGAN_x4plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', 4),
    'RealESRGAN_x4plus_anime_6B': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth', 4),
    'RealESRGAN_x2plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth', 2),
    'ESRGAN_SRx4_DF2KOST': (
        'https://github.com/xinntao/ESRGAN/releases/download/v0.1.1/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth', 4),
}

MODELS_DIR = '/content/models'
os.makedirs(MODELS_DIR, exist_ok=True)

def download_model(model_name):
    url, _ = MODEL_URLS[model_name]
    model_path = os.path.join(MODELS_DIR, f'{model_name}.pth')
    if not os.path.exists(model_path):
        print(f'⬇️  Завантаження моделі {model_name}...')
        urllib.request.urlretrieve(url, model_path)
        print(f'✅ Модель завантажено')
    else:
        print(f'✅ Модель вже є: {model_name}')
    return model_path

def build_upsampler(model_name, outscale):
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    model_path = download_model(model_name)
    _, model_scale = MODEL_URLS[model_name]
    num_block = 6 if 'anime_6B' in model_name else 23
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=num_block, num_grow_ch=32, scale=model_scale)
    return RealESRGANer(
        scale=model_scale, model_path=model_path, model=model,
        tile=400, tile_pad=10, pre_pad=0,
        half=torch.cuda.is_available(),
    )

def auto_resize_for_speed(img_path, outscale, max_pixels_map):
    """Зменшує зображення якщо воно більше ліміту для даного масштабу."""
    img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        return img_path
    h, w = img.shape[:2]
    total_px = w * h
    max_px = max_pixels_map.get(outscale, 3_000_000)
    if total_px <= max_px:
        return img_path
    ratio = math.sqrt(max_px / total_px)
    new_w, new_h = int(w * ratio), int(h * ratio)
    print(f'⚙️  Авто-зменшення: {w}×{h} → {new_w}×{new_h} px (ліміт {max_px//1_000_000:.0f} Mpx для x{outscale})')
    resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
    resized_path = img_path.replace('.', '_resized.')
    cv2.imwrite(resized_path, resized)
    return resized_path

def enhance_image(input_path, output_path, model_name, outscale, max_pixels_map, timeout=30):
    input_path = auto_resize_for_speed(input_path, outscale, max_pixels_map)
    img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'Не вдалося прочитати: {input_path}')
    h_orig, w_orig = img.shape[:2]
    print(f'📐 Вхід: {w_orig}×{h_orig} px  |  Модель: {model_name}  |  Масштаб: ×{outscale}')
    upsampler = build_upsampler(model_name, outscale)

    # ── Прогрес-бар ──
    _pb = widgets.IntProgress(
        value=0, min=0, max=100,
        description='⏳  0%',
        bar_style='info',
        style={'bar_color': '#4a90e2', 'description_width': '60px'},
        layout=widgets.Layout(width='100%', height='30px'),
    )
    display(_pb)

    _real_stdout = sys.stdout

    class _TileCapture:
        """Перехоплює рядки "Tile X/Y" та оновлює прогрес-бар; решту пропускає."""
        def write(self, text):
            m = re.search(r'Tile\s+(\d+)/(\d+)', text)
            if m:
                cur, tot = int(m.group(1)), int(m.group(2))
                pct = int(cur / tot * 100)
                _pb.value = pct
                _pb.description = f'🔄 {pct:>3}%'
            else:
                _real_stdout.write(text)
        def flush(self):
            _real_stdout.flush()
        def __getattr__(self, name):
            return getattr(_real_stdout, name)

    sys.stdout = _TileCapture()

    _result = [None]
    _error  = [None]

    def _run():
        try:
            out, _ = upsampler.enhance(img, outscale=outscale)
            _result[0] = out
        except Exception as e:
            _error[0] = e

    t = time.time()
    _thread = threading.Thread(target=_run, daemon=True)
    _thread.start()
    _thread.join(timeout=timeout)

    sys.stdout = _real_stdout  # завжди відновлюємо

    if _thread.is_alive():
        _pb.bar_style = 'danger'
        _pb.description = '⏱️ Timeout'
        raise TimeoutError(
            f'⏱️  Обробка перевищила {timeout} сек!\n'
            f'   Спробуй: менший масштаб (x2) або менше фото (до {_max_file_mb} MB).'
        )
    if _error[0]:
        _pb.bar_style = 'danger'
        _pb.description = '❌ Помилка'
        raise _error[0]

    _pb.value = 100
    _pb.description = '✅ 100%'
    _pb.bar_style = 'success'

    elapsed = time.time() - t
    output_img = _result[0]
    h_out, w_out = output_img.shape[:2]
    print(f'✅ Готово за {elapsed:.1f} сек  →  {w_out}×{h_out} px')
    cv2.imwrite(output_path, output_img)
    return {
        'orig_size': (w_orig, h_orig),
        'out_size':  (w_out, h_out),
        'elapsed':   elapsed,
        'output_path': output_path,
    }

# ════════════════════════════════════════════════
# 🔎 ПЕРЕВІРКИ ПЕРЕД КРОКОМ 3
# ════════════════════════════════════════════════
_ok = True

# 1) Кроки 1 і 2 виконано?
for _var, _name in [('upload_btn', 'Крок 2'), ('OUTPUT_DIR', 'Крок 2'), ('torch', 'Крок 1')]:
    if _var not in dir() and _var not in sys.modules:
        print(f'❌ {_name} не виконано! Запусти кроки по порядку.')
        _ok = False

if _ok:
    print('✅ Кроки 1 і 2 виконано')

# 2) Фото завантажено?
if _ok:
    if not upload_btn.value:
        print('❌ Фото не вибрано! Поверніться до Кроку 2 і натисніть "📸 Вибрати фото".')
        _ok = False
    else:
        _fname = list(upload_btn.value.keys())[0]
        _fdata = list(upload_btn.value.values())[0]
        _size_mb = len(_fdata['content']) / 1024**2
        print(f'✅ Фото: {_fname} ({_size_mb:.1f} MB)')

        # 3) Ліміт розміру файлу
        if _size_mb > _max_file_mb:
            print(f'❌ Файл {_size_mb:.1f} MB > ліміту {_max_file_mb} MB — обери менше фото.')
            _ok = False
        elif _size_mb == 0:
            print('❌ Файл порожній — спробуй завантажити інше фото.')
            _ok = False

        # 4) Перевірка зображення + попередження про пікселі
        if _ok:
            try:
                _img_check = Image.open(io.BytesIO(_fdata['content']))
                _w, _h = _img_check.size
                _total_px = _w * _h
                _scale_val = scale_options.value
                _limit_px = _max_pixels.get(_scale_val, 3_000_000)
                print(f'✅ Розмір: {_w}×{_h} px ({_total_px/1_000_000:.1f} Mpx) | режим: {_img_check.mode}')
                if _w < 50 or _h < 50:
                    print('⚠️  Зображення дуже маленьке (< 50px) — результат може бути поганим.')
                if _total_px > _limit_px:
                    _ratio = math.sqrt(_limit_px / _total_px)
                    _nw, _nh = int(_w * _ratio), int(_h * _ratio)
                    print(f'⚙️  {_total_px/1_000_000:.1f} Mpx > ліміту {_limit_px/1_000_000:.0f} Mpx для x{_scale_val}.')
                    print(f'   Буде авто-зменшено до {_nw}×{_nh} перед обробкою.')
                else:
                    _est_sec = max(2, int(_total_px / _limit_px * _proc_timeout))
                    print(f'✅ Розмір в нормі — орієнтовний час: ~{_est_sec} сек')
            except Exception as _e:
                print(f'❌ Не вдалося відкрити зображення: {_e}')
                _ok = False

# 5) GPU пам'ять
if _ok and 'torch' in sys.modules:
    import torch as _t
    if _t.cuda.is_available():
        _t.cuda.empty_cache()
        _free_vram = (_t.cuda.get_device_properties(0).total_memory
                      - _t.cuda.memory_allocated()) / 1024**3
        print(f'✅ Вільна VRAM: {_free_vram:.1f} GB')
        if _free_vram < 1.5:
            print('⚠️  Мало VRAM — Runtime → Restart runtime → повтори з Кроку 1.')
    else:
        print('⚠️  GPU недоступний — обробка на CPU (повільно).')

# 6) Місце на диску
if _ok:
    try:
        _, _, _free_d = shutil.disk_usage('/content')
        if _free_d / 1024**3 < 0.5:
            print('❌ Менше 500 MB на диску — немає місця для результату.')
            _ok = False
        else:
            print(f'✅ Диск: {_free_d/1024**3:.1f} GB вільно')
    except Exception:
        pass

if not _ok:
    raise SystemExit('❌ Усунь помилки вище перед запуском Кроку 3.')

print(f'\n✅ Всі перевірки пройдено — починаємо обробку (таймаут {_proc_timeout} сек)!\n')

# ════════════════════════════════════════════════
# 3B. ЗАПУСК ОБРОБКИ
# ════════════════════════════════════════════════
from google.colab import files

selected_model = model_options.value
selected_scale = scale_options.value

TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

for fname, fdata in upload_btn.value.items():
    input_path = f'{TEMP_DIR}/{fname}'
    with open(input_path, 'wb') as f:
        f.write(fdata['content'])

    base, _ = os.path.splitext(fname)
    result_name  = f'{base}_enhanced_x{selected_scale}.png'
    local_output = f'{TEMP_DIR}/{result_name}'
    drive_output = f'{OUTPUT_DIR}/{result_name}'

    print('=' * 55)
    print(f'🎨 PhotoEnhance — {fname}')
    print('=' * 55)

    try:
        stats = enhance_image(
            input_path, local_output, selected_model, selected_scale,
            _max_pixels, timeout=_proc_timeout
        )
    except TimeoutError as _te:
        print(str(_te))
        raise SystemExit('❌ Обробка зупинена через таймаут.')

    if DRIVE_AVAILABLE and drive_output != local_output:
        shutil.copy2(local_output, drive_output)
        print(f'☁️  Збережено на Drive: {drive_output}')
    else:
        print('💾 Drive не підключено — скачується нижче')

# ════════════════════════════════════════════════
# 3C. BEFORE / AFTER + СТАТИСТИКА
# ════════════════════════════════════════════════
orig_img = Image.open(input_path).convert('RGB')
enh_img  = Image.open(local_output).convert('RGB')

# Зменшуємо для відображення якщо дуже великі
_MAX_DISP = 2000
if max(enh_img.size) > _MAX_DISP:
    enh_img.thumbnail((_MAX_DISP, _MAX_DISP), Image.LANCZOS)
if max(orig_img.size) > _MAX_DISP:
    orig_img.thumbnail((_MAX_DISP, _MAX_DISP), Image.LANCZOS)

fig = plt.figure(figsize=(10, 5), facecolor='#1a1a2e')
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.03)

ax1 = fig.add_subplot(gs[0])
ax1.imshow(orig_img)
ax1.set_title(f'ОРИГІНАЛ\n{stats["orig_size"][0]}×{stats["orig_size"][1]} px',
              color='white', fontsize=10, pad=8)
ax1.axis('off')
ax1.text(0.02, 0.02, '◀ BEFORE', transform=ax1.transAxes,
         color='white', fontsize=9, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#e74c3c', alpha=0.85))

ax2 = fig.add_subplot(gs[1])
ax2.imshow(enh_img)
ax2.set_title(f'ПОКРАЩЕНО (×{selected_scale})\n{stats["out_size"][0]}×{stats["out_size"][1]} px',
              color='white', fontsize=10, pad=8)
ax2.axis('off')
ax2.text(0.02, 0.02, 'AFTER ▶', transform=ax2.transAxes,
         color='white', fontsize=9, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#2ecc71', alpha=0.85))

fig.suptitle(f'🎨 PhotoEnhance — Real-ESRGAN  |  {stats["elapsed"]:.1f}s',
             color='white', fontsize=12, fontweight='bold', y=1.01)

comparison_path = f'{TEMP_DIR}/comparison.png'
plt.savefig(comparison_path, dpi=100, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

pixel_increase = (stats['out_size'][0] * stats['out_size'][1]) / \
                 (stats['orig_size'][0] * stats['orig_size'][1])
print('\n' + '=' * 55)
print('📊 СТАТИСТИКА')
print('=' * 55)
print(f'  📥 Оригінал:  {stats["orig_size"][0]:>5} × {stats["orig_size"][1]} px')
print(f'  📤 Результат: {stats["out_size"][0]:>5} × {stats["out_size"][1]} px')
print(f'  🔍 Масштаб:   ×{selected_scale}  |  📈 Пікселів: ×{pixel_increase:.1f}')
print(f'  ⏱️  Час:        {stats["elapsed"]:.1f} сек  |  🤖 {selected_model}')
print('=' * 55)

# ════════════════════════════════════════════════
# 3D. СКАЧУВАННЯ (mobile-friendly)
# ════════════════════════════════════════════════
def make_download_link(filepath, label, mime='image/png'):
    with open(filepath, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    filename = os.path.basename(filepath)
    return (
        f'<a href="data:{mime};base64,{b64}" download="{filename}" '
        f'style="display:inline-block;margin:8px 4px;padding:12px 20px;'
        f'background:#2ecc71;color:#fff;font-weight:bold;font-size:15px;'
        f'border-radius:8px;text-decoration:none;min-width:200px;text-align:center;">'
        f'⬇️ {label}</a>'
    )

display(HTML("""
<div style="background:#f0f4ff;border-radius:8px;padding:12px;margin-top:10px;
            border-left:4px solid #4a90e2;font-family:sans-serif;">
  <b>📱 На телефоні:</b> натисни кнопку нижче → браузер збереже файл.<br>
  <small>Якщо не спрацює — натисни й утримуй зображення вище → "Зберегти зображення".</small>
</div>
""" + make_download_link(local_output, 'Зберегти покращене фото')
   + make_download_link(comparison_path, 'Зберегти порівняння')))

# Резервний варіант через google.colab (десктоп)
try:
    files.download(local_output)
    files.download(comparison_path)
except Exception:
    pass

print('\n✅ Готово!')
